# IndiVoice-DeepASR: Extended Training (2000 -> 4000 Steps)

This notebook resumes training from the Step 2000 checkpoint and extends it to Step 4000 for maximum accuracy.

In [ ]:
import os, shutil

# 1. Configuration
# Note: Replace with your actual HF token manually or via Kaggle Secrets
os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN_HERE'
MODEL_REPO = "purvansh01/whisper-indian-lora"
OUTPUT_DIR = "/kaggle/working/models/whisper-indian-lora"
REPO_DIR = "/kaggle/working/IndiVoice-DeepASR"

print("[LOG] Environment and paths initialized.")

In [ ]:
# 2. Fast Repository Clone
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("[LOG] Cleaned old repo folder.")

print("[LOG] Starting Git Clone...")
!git clone --depth 1 https://github.com/purvanshjoshi/IndiVoice-DeepASR.git {REPO_DIR}
print("[LOG] Git Clone Finished.")

%cd {REPO_DIR}

In [ ]:
# 3. Install Dependencies
print("[LOG] Starting pip installations (this usually takes 2-3 minutes)...")
# Explicitly upgrade torchao to fix peft compatibility issues on Kaggle
!pip install -q -U torchao
!pip install -q -r requirements.txt
!pip install -q -U accelerate bitsandbytes peft transformers datasets
print("[LOG] All dependencies installed.")

In [ ]:
# 4. Download Checkpoint from Hub
from huggingface_hub import snapshot_download

print(f"[LOG] Downloading Step 2000 checkpoint from {MODEL_REPO}...")
snapshot_download(
    repo_id=MODEL_REPO,
    local_dir=OUTPUT_DIR,
    allow_patterns=["last-checkpoint/*", "adapter_model.safetensors", "adapter_config.json", "preprocessor_config.json"]
)

# Rename 'last-checkpoint' to 'checkpoint-2000'
src_path = os.path.join(OUTPUT_DIR, "last-checkpoint")
dst_path = os.path.join(OUTPUT_DIR, "checkpoint-2000")

if os.path.exists(src_path):
    if os.path.exists(dst_path): shutil.rmtree(dst_path)
    os.rename(src_path, dst_path)
    print(f"[SUCCESS] Prepared checkpoint at {dst_path}")
else:
    print("[WARNING] Could not find 'last-checkpoint'. Starting fresh if needed.")

In [ ]:
# 5. Run Environment Setup
!chmod +x /kaggle/working/IndiVoice-DeepASR/kaggle/setup_kaggle.sh
!/kaggle/working/IndiVoice-DeepASR/kaggle/setup_kaggle.sh

In [ ]:
# 6. Launch Training (2000 -> 4000)
print("[LOG] Resuming Training to Step 4000...")
!accelerate launch src/train.py \
    --model_name "openai/whisper-medium" \
    --output_dir "/kaggle/working/models/whisper-indian-lora" \
    --batch_size 8 \
    --grad_accum 2 \
    --learning_rate 1e-4 \
    --max_steps 4000 \
    --hub_model_id "purvansh01/whisper-indian-lora"